# 04 - RAG Local (Retrieval Augmented Generation)

En este notebook aprenderemos a:
- Cargar y dividir documentos en fragmentos
- Crear embeddings locales con Ollama
- Almacenar vectores en FAISS (base de datos vectorial local)
- Hacer preguntas sobre documentos usando RAG
- Integrar RAG como herramienta de un agente

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="gemma3:12b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## 4.1 ¿Qué son los embeddings?

Un embedding convierte texto en un vector numérico que captura su significado. Textos similares producen vectores cercanos.

In [ ]:
# Crear embeddings de ejemplo
textos = [
    "El gato duerme en el sofá",
    "El felino descansa en el mueble",
    "Python es un lenguaje de programación",
]

vectores = embeddings.embed_documents(textos)

print(f"Dimensiones del vector: {len(vectores[0])}")
print(f"Primeros 5 valores del primer vector: {vectores[0][:5]}")

# Calcular similitud entre vectores
import numpy as np

def similitud_coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nSimilitud 'gato/sofá' vs 'felino/mueble': {similitud_coseno(vectores[0], vectores[1]):.3f}")
print(f"Similitud 'gato/sofá' vs 'Python':          {similitud_coseno(vectores[0], vectores[2]):.3f}")

## 4.2 Crear documentos de ejemplo

Vamos a crear unos documentos de texto para practicar RAG.

In [ ]:
from langchain_core.documents import Document

# Documentos de ejemplo sobre temas colombianos
documentos = [
    Document(
        page_content="""El Acuerdo de Paz de 2016 fue firmado entre el gobierno colombiano y las FARC-EP 
        en el Teatro Colón de Bogotá el 24 de noviembre de 2016. Este acuerdo buscó poner fin a más 
        de 50 años de conflicto armado interno. Incluye puntos sobre reforma rural integral, 
        participación política, fin del conflicto, solución al problema de drogas ilícitas, 
        víctimas del conflicto, e implementación y verificación.""",
        metadata={"fuente": "historia_paz.txt", "tema": "paz"}
    ),
    Document(
        page_content="""La Unidad de Búsqueda de Personas dadas por Desaparecidas (UBPD) es una entidad 
        del Estado colombiano creada por el Acuerdo de Paz. Su mandato es dirigir, coordinar y 
        contribuir a la implementación de acciones humanitarias de búsqueda y localización de 
        personas dadas por desaparecidas en el contexto y en razón del conflicto armado.""",
        metadata={"fuente": "ubpd.txt", "tema": "institucional"}
    ),
    Document(
        page_content="""La biodiversidad de Colombia es una de las más ricas del mundo. El país alberga 
        cerca del 10% de las especies del planeta. Tiene más de 1.900 especies de aves (el país 
        con más aves del mundo), 4.270 especies de orquídeas, y es el segundo país más biodiverso 
        del mundo después de Brasil.""",
        metadata={"fuente": "biodiversidad.txt", "tema": "naturaleza"}
    ),
    Document(
        page_content="""El café colombiano es reconocido mundialmente por su calidad. Colombia es el 
        tercer productor mundial de café. La Federación Nacional de Cafeteros, creada en 1927, 
        representa a más de 540.000 familias cafeteras. El Eje Cafetero, declarado Patrimonio 
        de la Humanidad por la UNESCO, incluye los departamentos de Caldas, Quindío y Risaralda.""",
        metadata={"fuente": "cafe.txt", "tema": "economía"}
    ),
    Document(
        page_content="""La justicia transicional en Colombia incluye la Jurisdicción Especial para la 
        Paz (JEP), la Comisión de la Verdad, y la Unidad de Búsqueda de Personas dadas por 
        Desaparecidas (UBPD). Estas tres entidades conforman el Sistema Integral de Verdad, 
        Justicia, Reparación y No Repetición (SIVJRNR), creado por el Acuerdo de Paz de 2016.""",
        metadata={"fuente": "justicia_transicional.txt", "tema": "justicia"}
    ),
]

print(f"Creados {len(documentos)} documentos de ejemplo")

## 4.3 Dividir documentos en fragmentos (Text Splitting)

Para documentos largos, los dividimos en fragmentos más pequeños que el modelo pueda procesar.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # Tamaño máximo de cada fragmento (caracteres)
    chunk_overlap=50,     # Superposición entre fragmentos
    separators=["\n\n", "\n", ". ", " "],  # Prioridad de separación
)

fragmentos = splitter.split_documents(documentos)

print(f"Documentos originales: {len(documentos)}")
print(f"Fragmentos generados: {len(fragmentos)}")
print(f"\nEjemplo de fragmento:")
print(f"  Contenido: {fragmentos[0].page_content[:100]}...")
print(f"  Metadata: {fragmentos[0].metadata}")

## 4.4 Crear base de datos vectorial con FAISS

FAISS almacena los vectores localmente y permite buscar por similitud.

In [ ]:
from langchain_community.vectorstores import FAISS

# Crear la base de datos vectorial en memoria
vectorstore = FAISS.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    
)

print(f"Base de datos creada con {vectorstore.index.ntotal} fragmentos")

## 4.5 Búsqueda por similitud

In [ ]:
# Buscar fragmentos relevantes
pregunta = "¿Qué es la UBPD?"
resultados = vectorstore.similarity_search(pregunta, k=3)

print(f"Pregunta: {pregunta}\n")
for i, doc in enumerate(resultados):
    print(f"--- Resultado {i+1} (fuente: {doc.metadata.get('fuente', 'N/A')}) ---")
    print(doc.page_content[:200])
    print()

## 4.6 RAG Chain: pregunta → busca → responde

Ahora combinamos la búsqueda con el LLM para responder preguntas basándose en los documentos.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def formatear_contexto(docs):
    """Convierte una lista de documentos en un string de contexto."""
    return "\n\n".join(
        f"[Fuente: {d.metadata.get('fuente', 'N/A')}]\n{d.page_content}"
        for d in docs
    )


prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """Responde la pregunta basándote ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, di que no tienes información suficiente.
Cita la fuente cuando sea posible.

Contexto:
{contexto}"""),
    ("human", "{pregunta}"),
])

# Crear el retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Chain RAG completa
rag_chain = (
    {"contexto": retriever | formatear_contexto, "pregunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

# Probar
preguntas = [
    "¿Qué entidades conforman el Sistema Integral de Verdad?",
    "¿Cuántas especies de aves tiene Colombia?",
    "¿Cuándo se firmó el Acuerdo de Paz?",
]

for p in preguntas:
    print(f"\nP: {p}")
    print(f"R: {rag_chain.invoke(p)}")
    print("-" * 60)

## 4.7 RAG con documentos PDF reales

Ahora carguemos un PDF real. Coloca un archivo PDF en la carpeta `data/` para probarlo.

In [ ]:
import os
from pathlib import Path

# Crear carpeta data si no existe
Path("../data").mkdir(exist_ok=True)

# Verificar si hay PDFs
pdfs = list(Path("../data").glob("*.pdf"))

if pdfs:
    from langchain_community.document_loaders import PyMuPDFLoader

    # Cargar el primer PDF encontrado
    loader = PyMuPDFLoader(str(pdfs[0]))
    docs_pdf = loader.load()

    print(f"PDF cargado: {pdfs[0].name}")
    print(f"Páginas: {len(docs_pdf)}")
    print(f"\nPrimeros 300 caracteres de la página 1:")
    print(docs_pdf[0].page_content[:300])

    # Dividir y agregar al vectorstore
    fragmentos_pdf = splitter.split_documents(docs_pdf)
    vectorstore.add_documents(fragmentos_pdf)
    print(f"\nAgregados {len(fragmentos_pdf)} fragmentos al vectorstore")
else:
    print("No hay PDFs en data/. Coloca un PDF ahí para probarlo.")
    print("Puedes continuar con los documentos de ejemplo que ya cargamos.")

## Ejercicio

1. Agrega 3 documentos más al vectorstore sobre un tema de tu interés
2. Modifica el prompt RAG para que responda en formato de viñetas
3. Implementa una búsqueda con filtro por metadata (ej: solo documentos de tema "paz")

In [ ]:
# Tu código aquí
